# Test the code from file `ent_meas.py`

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np

import matplotlib.pyplot as plt

# Local importations
from moments.ent_meas import compute_concurrence, compute_binary_entropy, compute_eof

# Binary entropy

- `compute_binary_entropy`: Takes as input a real number or numpy array with elements $x_i \in [0, 1]$. It returns the binary cross entropy $S(x_i)$.

To check the expected behavour, we compute the binary cross entropy across the hole domain.

In [ ]:
# Define the hole binary entropy domain.
x = np.linspace(0, 1, 1000)

# Compute binary cross entorpy.
S = compute_binary_entropy(x)

# Plot results.
fig, ax = plt.subplots()
ax.plot(x, S)
ax.set(xlabel="x", ylabel="S(x)", title="Binary entropy")
ax.grid(True)
plt.show()

# Concurrence and entanglement of formation

- `compute_concurrence`: takes as input a numpy array representing a quantum density matrix and outputs a float being its concurrence.
- `compute_eof`: takes as input either a numpy array representiing a quantum density matrix or a float/numpy array representing a value/sequence of concurrences. It returns a float/numpy array representing a value/sequence of values of the entanglement of formation.

To check that the entanglement of formation works, we check the it reduces to the Von Neumann Entropy for all pure states.

To check that the concurrence works, we check that both methods of computing the entanglement of formation (with and without the concurrence) agree on the same result.

In [ ]:
def compute_pure_LU_ent_classes(theta: float) -> np.ndarray:
    """
    LU entanglement classes in thw two-qubit case can be completely determined by a single parameter (theta in this case).

    Parameters
    ----------
    theta : float
        Parameter indicating the LU entanglement class of the state that we want to compute.
    
    Returns
    -------
    np.ndarray
        Representative state of the infinite pure states in the determined entanglement class.
    """
    rho = np.zeros((4, 4), dtype=complex)
    rho[0, 0] = np.cos(theta)**2
    rho[3, 3] = np.sin(theta)**2
    rho[0, 3] = rho[3, 0] = np.cos(theta)*np.sin(theta)
    return rho

def compute_ent_entropy_LU_ent_classes(theta: float | np.ndarray, tol: float = 1e-12) -> float | np.ndarray:
    """
    In two qubit pure states, the entanglement of formation reduces to the Von Neumann entropy.

    Parameters
    ----------
    theta : float | np.ndarray
        Parameter indicating the LU entanglement class of the state that we want to compute.
    tol : float
        Tolerance to avoid the indeterminacy of the logarithm in the Von Neumann entropy.
    
    Returns
    -------
    float | np.ndarray
        Value of the Von Neumann entropy correspondig to the LU entanglement class determined by theta.
    """
    theta = np.asarray(theta)
    result = np.zeros_like(theta, dtype=np.float64)

    lambda1_sq = np.cos(theta)**2
    lambda2_sq = np.sin(theta)**2
    
    mask1 = lambda1_sq > tol
    mask2 = lambda2_sq > tol
    lambda1_valid = lambda1_sq[mask1]
    lambda2_valid = lambda2_sq[mask2]
    
    result[mask1] += -lambda1_valid * np.log2(lambda1_valid)
    result[mask2] += -lambda2_valid * np.log2(lambda2_valid)

    return result.item() if result.ndim == 0 else result

In [ ]:
# Generate paremeter values for all entanglement classes
theta_values = np.linspace(0, np.pi/4, 1000)

# Compute Von Neumann entropy
S = compute_ent_entropy_LU_ent_classes(theta_values)

# Compute EoF and concurrence
concurrences = []
EoF = []
EoF_C = []
for theta in theta_values:
    rho = compute_pure_LU_ent_classes(theta)
    C = compute_concurrence(rho)
    E = compute_eof(rho)
    E_C = compute_eof(C = C)
    concurrences.append(C)
    EoF.append(E)
    EoF_C.append(E_C)

In [ ]:
# Compare EoF, concurrence and Von Neumann entropy
fig, ax = plt.subplots()
ax.plot(theta_values, concurrences, label="Concurrence")
ax.plot(theta_values, EoF, label="EoF")
ax.plot(theta_values, EoF_C, label="EoF from C")
ax.plot(theta_values, S, label="Von Neumann Entropy")
ax.set(xlabel=r"$\theta$", ylabel="Entanglement measures", title="Entanglement measures for pure LU entanglement classes",
       xticks=np.arange(0, np.pi/4 + 0.01, np.pi/16), xticklabels=[r"$0$", r"$\frac{\pi}{16}$", r"$\frac{\pi}{8}$", r"$\frac{3\pi}{16}$", r"$\frac{\pi}{4}$"])
ax.legend()
ax.grid(True)
plt.show()